## Example 1 — the import, the version and LMCache's compiled modules

In [ ]:
import importlib, importlib.metadata, importlib.util
import lmcache
for name in ('lmcache.lmcache_native', 'lmcache.lmcache_redis', 'lmcache.lmcache_fs'):
    module = importlib.import_module(name)
    assert module.__file__.endswith('.so'), module.__file__
cuda_ops = importlib.import_module('lmcache.cuda_ops')
assert cuda_ops.__file__.endswith('.so'), cuda_ops.__file__
assert callable(cuda_ops.multi_layer_kv_transfer) and callable(cuda_ops.calculate_cdf), dir(cuda_ops)
print('OK -- lmcache', importlib.metadata.version('lmcache'))

## Example 2 — the C++ bitmap that tracks which tokens of a chunk are cached

In [ ]:
# Adapted from tests/v1/lmcache_native/test_bitmap.py (LMCache v0.5.5, Apache-2.0).
from lmcache.lmcache_native import Bitmap
bitmap = Bitmap(128)
for position in (0, 5, 64, 127):
    bitmap.set(position)
assert bitmap.test(5) and bitmap.test(127) and not bitmap.test(6)
assert bitmap.popcount() == 4, bitmap.popcount()
bitmap.clear(5)
assert not bitmap.test(5) and bitmap.popcount() == 3
print('OK -- Bitmap(128): 4 bits set, one cleared, popcount', bitmap.popcount())

## Example 3 — the address manager that hands out aligned blocks of the cache's memory pool

In [ ]:
# Adapted from tests/v1/test_address_manager.py (LMCache v0.5.5, Apache-2.0). Pure Python, no GPU.
from lmcache.v1.memory_management import AddressManager
manager = AddressManager(4096 * 10)  # a 40 KiB pool, 4 KiB alignment
first, first_size = manager.allocate(100)       # rounds up to one page
second, second_size = manager.allocate(5000)    # rounds up to two pages
assert (first_size, second_size) == (4096, 8192), (first_size, second_size)
assert second == first + first_size  # the blocks are laid out one after the other
assert manager.get_free_size() == 4096 * 10 - 3 * 4096
manager.free(first, first_size)
manager.free(second, second_size)
assert manager.get_free_size() == 4096 * 10 and manager.check_consistency()  # freed blocks merge back
print('OK -- allocated', first_size + second_size, 'bytes in two aligned blocks and merged them back')

## Example 4 — the generated gRPC lookup-service messages and numba chunk hashing

In [ ]:
# Written from lmcache/v1/multiprocess (LMCache v0.5.5). Importing the generated protobuf and gRPC modules is the check.
import numpy as np
from lmcache.v1.multiprocess.transport.grpc_impl._proto_gen import lookup_service_pb2 as pb, lookup_service_pb2_grpc as pbg
from lmcache.v1.multiprocess import token_hasher
message_name = next(iter(pb.DESCRIPTOR.message_types_by_name))
message = getattr(pb, message_name)()
assert type(message).FromString(message.SerializeToString()) == message
assert any(name.endswith('Stub') for name in dir(pbg)) and pb.DESCRIPTOR.services_by_name
tokens = np.arange(64, dtype=np.uint64)
hashes = token_hasher.chunk_hash_windows_numba(tokens, 16, np.uint64(31))
assert len(hashes) > 0 and len(set(hashes.tolist())) == len(hashes)
print('OK --', message_name, 'round-trips;', len(hashes), 'distinct chunk hashes from numba')